In [1]:
# new_selector.xpath("//a[@class='inline-block group']/@href").getall() //get all the links on a page
# new_selector.xpath("//h1[@class='font-bold type-preset-2']/text()").get() //name of the faculty 
# new_selector.xpath("//dl[@class='mt-8']//text()").getall() // department of the faculty
# new_selector.xpath("//div[@class='border-t border-slate-100 text-blue-400']//a/@href").getall() //link to personal page
# new_selector.xpath("//div[@class='gutenberg-editor']/p/text()").getall() // info of the faculty
# new_selector.xpath("//div[@class='pagination text-center']/a/@href").get() // next page

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from scrapy import Selector
driver_path = r'.\chromedriver.exe'
brave_path = r'C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe'

service = Service(driver_path)
option = webdriver.ChromeOptions()
option.binary_location = brave_path
browser = webdriver.Chrome(service=service, options=option)
browser.get("https://datascience.columbia.edu/people-type/faculty/")
new_selector = Selector(text=browser.page_source)

In [ ]:
with open('DSI Faculty.txt', "w", encoding="utf-8") as w:
    while next_page:=new_selector.xpath("//div[@class='pagination text-center']/a[@class='next page-numbers']/@href").get():
        for faculty in new_selector.xpath("//a[@class='inline-block group']/@href").getall():
            browser.get(faculty)
            info_selector = Selector(text=browser.page_source)
            name = info_selector.xpath("//h1[@class='font-bold type-preset-2']/text()").get() # name of the faculty 
            dept = info_selector.xpath("//dl[@class='mt-8']//text()").getall() # department of the faculty
            links = info_selector.xpath("//div[@class='border-t border-slate-100 text-blue-400']//a/@href").getall() # link to personal page
            info = info_selector.xpath("//div[@class='gutenberg-editor']/p/text()").getall() # info of the faculty
            w.write(f'{name}\n')
            for line in dept:
                w.write(f"{line.strip()}\n")
            for link in links:
                w.write(f'\n{link}')
            w.write('\n\n')
            for i in info:
                w.write(f"{i}")
            w.write('\n\n\n==============================================================================================================================================\n\n\n')
        browser.get(next_page)
        print(next_page)
        new_selector = Selector(text=browser.page_source)

### Information Retrival

In [1]:
%%capture
!pip install chromadb --no-deps
!pip install sentence-transformers==5.1.2
!pip install transformers==4.57.1
!pip install langchain langchain-community langchain-chroma

In [22]:
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer
import chromadb
from rich.traceback import install; install()
import pandas as pd 
import os
from dotenv import load_dotenv
from huggingface_hub import login
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
load_dotenv() # load the variables from .env file
hf_token = os.environ.get("HF_TOKEN")

client = chromadb.PersistentClient(path="/content/drive/MyDrive/db/")
collection = client.get_or_create_collection(
    name="faculty_info",
    metadata={"hnsw:space": "cosine"}
)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    token=hf_token,
)

In [16]:
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto", # this will put the device on cuda by default
    dtype="auto", # keeps weights in 16 bit,
    token=hf_token,
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [19]:
sent_model_name = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(sent_model_name, device=device, token=hf_token)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
mpnet_tokenizer = AutoTokenizer.from_pretrained(sent_model_name, token=hf_token)

In [21]:
# load the data
faculty_df = pd.read_json("/content/dsi_faculty.json", encoding='latin8')
faculty_df.head()

,name,dept,links,info
0,Ryan Abernathey,Faculty of Arts and Sciences\nAssociate Profes...,"[mailto:rpa@ldeo.columbia.edu, https://raberna...",Ryan P. Abernathey is an Associate Professor o...
1,Paris Adkins-Jackson,Mailman School of Public Health\nAssistant Pro...,"[mailto:pa2629@cumc.columbia.edu, https://www....","Paris AJ Adkins-Jackson, PhD MPH is a multid..."
2,Anish Agarwal,Columbia Engineering\nAssistant Professor of I...,"[mailto:aa5194@columbia.edu, https://sites.goo...",Anishs research interests are in causal infer...
3,Shipra Agrawal,Columbia Engineering\nCyrus Derman Associate P...,"[mailto:sa3305@columbia.edu, http://www.columb...",Professor Shipra Agrawal is the Cyrus Derman A...
4,Sunil Agrawal,Columbia Engineering\nProfessor of Mechanical ...,"[mailto:sunil.agrawal@columbia.edu, http://roa...",Dr. Agrawal obtained a PhD degree in Mechanica...


In [32]:
def chunk_text_token_overlap(text, chunk_size=300, overlap=50):
    # encode sentence to tokens
    # add_special_tokens is set to false otherwise overlapping tokens will be 
    # different from the sentence because a special BOS/EOS token is added
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    start = 0
    end = chunk_size

    while start < len(tokens):
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)

        start = end - overlap
        end = start + chunk_size

    return chunks

def store_info():
    for idx, row in faculty_df.iterrows():
        # create chunks for the faculty information
        chunks = chunk_text_token_overlap(row['info'])

        # generate embeddings
        embeddings = embedder.encode(
            chunks,
            batch_size=32,
            convert_to_numpy=True,
            # all-mpnet-base-v2 normalizes by default;
            # we need normalized vector for comparision
            normalize_embeddings=False, # just setting flag explicitely 
        ) # embeddings shape: (n_chunks, 768)

        # add each chunk to the vector db
        for i, chunk in enumerate(chunks):
            collection.add(
                documents=[chunk],
                embeddings=[embeddings[i]],
                metadatas=[{
                    "name": row["name"],
                    "department": row["dept"],
                    "chunk_index": i,
                }],
                ids=[f"{row["name"]}_info_{i}"], # add some indentifier
            )

def generate_answer_mistral(prompt, max_tokens=256):
    # tokenize the prompt before feeding to the LLM
    # tokenizer returns a hg dictionary wrapper that has a .to method
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    output = model.generate(
        **inputs, # BatchEncoding object
        max_new_tokens=max_tokens, # stop after generating this many *new* toeksn
        do_sample=False,  # greedy decoding (deterministic, fastest, less delusions)
    ) 

    return tokenizer.decode(output[0], skip_special_tokens=True) # skip </s> <s>

def build_rag_prompt(query, retrieved_chunks):
    prompt = "You are a factual assistant. Use ONLY the evidence below.\n"
    prompt += "If the answer is not in the evidence, say 'NOT FOUND in the given evidence'\n\n"

    prompt += "EVIDENCE:\n"
    for i, chunk in enumerate(retrieved_chunks):
        text = chunk["text"]
        metadata = chunk["metadata"]
        prompt += f"[{i}] {text}\n"
        prompt += f"Metadata: {metadata}\n\n"
    
    prompt += f"QUESTION: {query}\n\n"
    prompt += "ANSWER (cite evidence like [0], [1], etc). Please provide only one concise answer.\n"

    return prompt

def retrive_and_answer(query, n_results=4):
    # create query vector from query text
    query_emb = embedder.encode(query)

    # retrive the results
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results
    )

    # stitch together fetched data to provide it for generation
    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({"text": doc, "metadata": meta}) 
    
    prompt = build_rag_prompt(query, retrieved)
    answer = generate_answer_mistral(prompt, max_tokens=256)
    return answer
def chunk_text_token_overlap(text, chunk_size=300, overlap=50):
    # encode sentence to tokens
    # add_special_tokens is set to false otherwise overlapping tokens will be 
    # different from the sentence because a special BOS/EOS token is added
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    start = 0
    end = chunk_size

    while start < len(tokens):
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)

        start = end - overlap
        end = start + chunk_size

    return chunks

def store_info():
    for idx, row in faculty_df.iterrows():
        # create chunks for the faculty information
        chunks = chunk_text_token_overlap(row['info'])

        # generate embeddings
        embeddings = embedder.encode(
            chunks,
            batch_size=32,
            convert_to_numpy=True,
            # all-mpnet-base-v2 normalizes by default;
            # we need normalized vector for comparision
            normalize_embeddings=False, # just setting flag explicitely 
        ) # embeddings shape: (n_chunks, 768)

        # add each chunk to the vector db
        for i, chunk in enumerate(chunks):
            collection.add(
                documents=[chunk],
                embeddings=[embeddings[i]],
                metadatas=[{
                    "name": row["name"],
                    "department": row["dept"],
                    "chunk_index": i,
                }],
                ids=[f"{row["name"]}_info_{i}"], # add some indentifier
            )

def generate_answer_mistral(prompt, max_tokens=256):
    # tokenize the prompt before feeding to the LLM
    # tokenizer returns a hg dictionary wrapper that has a .to method
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    output = model.generate(
        **inputs, # BatchEncoding object
        max_new_tokens=max_tokens, # stop after generating this many *new* toeksn
        do_sample=False,  # greedy decoding (deterministic, fastest, less delusions)
    ) 

    return tokenizer.decode(output[0], skip_special_tokens=True) # skip </s> <s>

def build_rag_prompt(query, retrieved_chunks):
    prompt = "You are a factual assistant. Use ONLY the evidence below.\n"
    prompt += "If the answer is not in the evidence, say 'NOT FOUND in the given evidence'\n\n"

    prompt += "EVIDENCE:\n"
    for i, chunk in enumerate(retrieved_chunks):
        text = chunk["text"]
        metadata = chunk["metadata"]
        prompt += f"[{i}] {text}\n"
        prompt += f"Metadata: {metadata}\n\n"
    
    prompt += f"QUESTION: {query}\n\n"
    prompt += "ANSWER (cite evidence like [0], [1], etc). Please provide only one concise answer.\n"

    return prompt

def retrive_and_answer(query, n_results=4):
    # create query vector from query text
    query_emb = embedder.encode(query)

    # retrive the results
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results
    )

    # stitch together fetched data to provide it for generation
    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({"text": doc, "metadata": meta}) 
    
    prompt = build_rag_prompt(query, retrieved)
    answer = generate_answer_mistral(prompt, max_tokens=256)
    return answer


In [24]:
store_info()

In [36]:
query = "Which professor should I cite in my application to Master's in Data Science program if I want to have the best chances? I am interested in Generative AI"
answers = retrive_and_answer(query)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [37]:
print(answers)

You are a factual assistant. Use ONLY the evidence below.
If the answer is not in the evidence, say 'NOT FOUND in the given evidence'

EVIDENCE:
[0] I am a professor in the Department of Computer Science at Columbia University. I am broadly interested in machine learning, artificial intelligence, statistics, neuroscience, and cognitive science. I am also the Director of the new NSF AI Institute for ARtificial and Natural Intelligence (ARNI).
Metadata: {'name': 'Richard Zemel', 'chunk_index': 0, 'department': 'Columbia Engineering\nTrianthe Dakolias Professor of Engineering and Applied Science and Professor of Computer Science'}

[1] Im an Assistant Professor of Biostatistics in the Mailman School of Public Health at Columbia University. My research focuses mostly on causal inference: developing statistical methods and machine learning tools to support inference about treatment effects, interventions, and policies. I often take a perspective informed by graphical models (e.g., DAGs/Bay